<a href="https://colab.research.google.com/github/pesco13/Python-para-Quimica/blob/main/notebooks/05_Cinetica_y_ajuste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 05 · Cinética química y ajuste no lineal

**Objetivos**
* Determinar el orden de una reacción a partir de datos de concentración frente a tiempo.
* Ajustar un modelo no lineal con `scipy.optimize.curve_fit` y obtener la incertidumbre de los parámetros.
* Comparar modelos de forma honesta, sin fiarse solo del R².
* Obtener la energía de activación con una gráfica de Arrhenius.

A diferencia de la práctica anterior, aquí el modelo **no** es una recta, y además hay que elegir entre varios modelos posibles. Esa elección es el verdadero trabajo del químico.

## 0. Los datos

Se sigue la desaparición de un reactivo a cuatro temperaturas. Ni el orden ni la energía de activación aparecen en los datos: hay que averiguarlos a partir de ellos.

In [ ]:
!wget -q -O generar_datos.py https://pesco13.github.io/Python-para-Quimica/datos/generar_datos.py

import os
print("descargado en", os.getcwd(), "-", os.path.getsize("generar_datos.py"), "bytes")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from generar_datos import cinetica, mi_asignacion

NUMERO = 7          # <<< cambie la semilla para obtener otra serie

print(mi_asignacion(NUMERO)["reaccion"])
serie = cinetica(NUMERO)
serie.head()

In [ ]:
temperaturas = sorted(serie["temperatura_C"].unique())
print("Temperaturas medidas (°C):", temperaturas)
print("Puntos por temperatura:", len(serie) // len(temperaturas))

Fíjese en algo del diseño experimental: a mayor temperatura la reacción se siguió durante menos tiempo. Es lo que se hace en el laboratorio, porque a 55 °C todo termina antes.

## 1. Las leyes de velocidad integradas

Para una reacción $A \rightarrow$ productos, la velocidad depende de la concentración elevada al orden $n$. Integrando se obtiene, para cada orden, una expresión distinta de la concentración frente al tiempo:

| Orden | Ley integrada | Forma que da una recta | Unidades de $k$ |
|---|---|---|---|
| 0 | $[A] = [A]_0 - kt$ | $[A]$ contra $t$ | mol L⁻¹ min⁻¹ |
| 1 | $[A] = [A]_0 e^{-kt}$ | $\ln[A]$ contra $t$ | min⁻¹ |
| 2 | $\dfrac{1}{[A]} = \dfrac{1}{[A]_0} + kt$ | $1/[A]$ contra $t$ | L mol⁻¹ min⁻¹ |

El método clásico consiste en probar las tres transformaciones y quedarse con la que dé la recta más convincente.

## 2. Probar los tres órdenes

Trabajemos primero con una sola temperatura, la más baja.

In [ ]:
from scipy.stats import linregress

T_baja = temperaturas[0]
g = serie[serie["temperatura_C"] == T_baja]
t = g["tiempo_min"].values
c = g["concentracion_M"].values

transformaciones = {
    "orden 0:  [A]":      c,
    "orden 1:  ln[A]":    np.log(c),
    "orden 2:  1/[A]":    1 / c,
}

for nombre, valores in transformaciones.items():
    r = linregress(t, valores)
    print(f"{nombre:18s}  R² = {r.rvalue**2:.5f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))

for ax, (nombre, valores) in zip(axes, transformaciones.items()):
    r = linregress(t, valores)
    ax.plot(t, valores, "o", color="#1b6ca8", ms=4)
    ax.plot(t, r.intercept + r.slope * t, "-", color="#d1495b", lw=1)
    ax.set_title(f"{nombre}\n$R^2$ = {r.rvalue**2:.4f}", fontsize=9)
    ax.set_xlabel("Tiempo (min)")

axes[0].set_ylabel("Valor transformado")
fig.suptitle(f"Las tres linealizaciones a {T_baja:.0f} °C", y=1.04)
fig.tight_layout()
fig.savefig("cinetica_ordenes.png", dpi=300, bbox_inches="tight")
plt.show()

El orden correcto es el de la transformación que da la recta más limpia. Mire los **puntos**, no solo el R²: en la transformación equivocada los puntos se curvan de forma sistemática, aunque el R² siga siendo alto.

> Anote el orden que eligió. Lo necesitará en el resto del notebook.

In [ ]:
ORDEN = 1          # <<< cambie a 0, 1 o 2 según lo que concluyó

print("Orden elegido:", ORDEN)

## 3. Por qué linealizar no es la mejor opción

Tomar logaritmos o inversos cambia el tamaño de los errores. El ruido del espectrofotómetro es aproximadamente constante en $[A]$, pero al calcular $1/[A]$ los puntos de concentración baja se disparan y pasan a dominar el ajuste. El resultado es una $k$ sesgada.

Lo correcto es ajustar el modelo **tal cual**, sin transformarlo. Eso es lo que hace `curve_fit`: busca los parámetros que minimizan la suma de cuadrados sobre los datos originales.

In [ ]:
from scipy.optimize import curve_fit

def orden_0(t, c0, k):
    return c0 - k * t

def orden_1(t, c0, k):
    return c0 * np.exp(-k * t)

def orden_2(t, c0, k):
    return c0 / (1 + k * c0 * t)

modelos = {0: orden_0, 1: orden_1, 2: orden_2}
modelo = modelos[ORDEN]

parametros, covarianza = curve_fit(modelo, t, c, p0=[c[0], 0.02])
c0_aj, k_aj = parametros
incertidumbres = np.sqrt(np.diag(covarianza))

print(f"[A]0 = {c0_aj:.4f} ± {incertidumbres[0]:.4f} mol/L")
print(f"k    = {k_aj:.5f} ± {incertidumbres[1]:.5f}")

`p0` es la estimación inicial. `curve_fit` la usa como punto de partida de la búsqueda; si es muy mala, el ajuste no converge. Una buena regla: para $[A]_0$ use el primer dato medido, y para $k$ un valor del orden de $1/t_{\text{final}}$.

La matriz `covarianza` contiene las varianzas de los parámetros en la diagonal. Su raíz cuadrada es la desviación estándar de cada parámetro: eso es lo que se reporta como incertidumbre.

## 4. Comparar los tres modelos de forma honesta

Como los tres modelos tienen el mismo número de parámetros, se pueden comparar directamente por la **suma de cuadrados residual**: el que deje los residuales más pequeños gana.

In [ ]:
print(f"{'modelo':10s} {'SCR':>12s} {'k ajustada':>14s}")
for n, f in modelos.items():
    try:
        p, _ = curve_fit(f, t, c, p0=[c[0], 0.02])
        scr = np.sum((c - f(t, *p))**2)
        print(f"orden {n:<4d} {scr:12.3e} {p[1]:14.5f}")
    except RuntimeError:
        print(f"orden {n:<4d} {'no converge':>12s}")

La suma de cuadrados del modelo correcto suele ser un orden de magnitud menor que la de los otros. Si dos modelos dan sumas parecidas, los datos no permiten distinguirlos: hay que medir durante más tiempo o con menos ruido, y así debe decirse en el informe.

## 5. Ajustar las cuatro temperaturas

In [ ]:
resultados = []
for T in temperaturas:
    g = serie[serie["temperatura_C"] == T]
    tt, cc = g["tiempo_min"].values, g["concentracion_M"].values
    p, cov = curve_fit(modelo, tt, cc, p0=[cc[0], 0.02])
    resultados.append({"T_C": T, "T_K": T + 273.15,
                       "k": p[1], "s_k": np.sqrt(np.diag(cov))[1]})

tabla_k = pd.DataFrame(resultados)
tabla_k

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
colores = plt.cm.viridis(np.linspace(0.12, 0.82, len(temperaturas)))

for color, T in zip(colores, temperaturas):
    g = serie[serie["temperatura_C"] == T]
    tt, cc = g["tiempo_min"].values, g["concentracion_M"].values
    p, _ = curve_fit(modelo, tt, cc, p0=[cc[0], 0.02])
    suave = np.linspace(0, tt.max(), 200)
    ax.plot(tt, cc, "o", color=color, ms=4)
    ax.plot(suave, modelo(suave, *p), "-", color=color, lw=1, label=f"{T:.0f} °C")

ax.set_xlabel("Tiempo (min)")
ax.set_ylabel("Concentración (mol L$^{-1}$)")
ax.set_title(f"Ajuste de orden {ORDEN} a las cuatro temperaturas")
ax.legend(frameon=False)
plt.show()

## 6. La ecuación de Arrhenius

La constante de velocidad depende de la temperatura según

$$k = A\,e^{-E_a/RT}$$

y tomando logaritmos queda una recta:

$$\ln k = \ln A - \frac{E_a}{R}\cdot\frac{1}{T}$$

La pendiente de $\ln k$ frente a $1/T$ vale $-E_a/R$. Aquí sí se linealiza, porque es la forma estándar en la que se reporta y porque los cuatro valores de $k$ tienen incertidumbres parecidas.

In [ ]:
R = 8.314462618e-3          # kJ / (mol K)

x_arr = 1 / tabla_k["T_K"].values
y_arr = np.log(tabla_k["k"].values)

arr = linregress(x_arr, y_arr)
Ea = -arr.slope * R
s_Ea = arr.stderr * R
A = np.exp(arr.intercept)

print(f"Ea = {Ea:.1f} ± {s_Ea:.1f} kJ/mol")
print(f"A  = {A:.3e}")
print(f"R² de la recta de Arrhenius = {arr.rvalue**2:.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_arr * 1000, y_arr, "o", color="#1b6ca8", ms=6)
xx = np.linspace(x_arr.min(), x_arr.max(), 50)
ax.plot(xx * 1000, arr.intercept + arr.slope * xx, "-", color="#d1495b", lw=1)

ax.set_xlabel("1000 / T (K$^{-1}$)")
ax.set_ylabel("ln k")
ax.set_title(f"Gráfica de Arrhenius: $E_a$ = {Ea:.0f} ± {s_Ea:.0f} kJ/mol")
fig.savefig("arrhenius.png", dpi=300, bbox_inches="tight")
plt.show()

Una energía de activación de 50 a 100 kJ/mol es lo habitual en reacciones en disolución. Valores mucho menores sugieren un proceso controlado por difusión; mucho mayores, la ruptura de un enlace fuerte en el paso determinante.

## Autocomprobación

In [ ]:
assert ORDEN in (0, 1, 2), "ORDEN debe valer 0, 1 o 2"
assert k_aj > 0, "la constante de velocidad no puede ser negativa"
assert len(tabla_k) == len(temperaturas), "falta ajustar alguna temperatura"
assert (tabla_k["k"].diff().dropna() > 0).all(), "k debe crecer con la temperatura: revise el orden elegido"
assert 10 < Ea < 250, "una Ea fuera de 10-250 kJ/mol indica un error en el ajuste"
print(f"Todo correcto. Orden {ORDEN}, Ea = {Ea:.1f} kJ/mol.")

## Ejercicios

**1.** Ajuste la temperatura más baja con el orden **equivocado** y compare la curva ajustada con los puntos. ¿Dónde se desvía más?

<details><summary><b>Ver solución</b></summary>

```python
otro = modelos[2 if ORDEN != 2 else 1]
p, _ = curve_fit(otro, t, c, p0=[c[0], 0.02])
fig, ax = plt.subplots()
ax.plot(t, c, "o")
suave = np.linspace(0, t.max(), 200)
ax.plot(suave, otro(suave, *p), "-")
plt.show()
```
El modelo equivocado suele acertar al principio y al final, y fallar en la zona media: por eso el R² no lo delata.
</details>

**2.** Calcule el tiempo de vida media a cada temperatura. Para orden 1 es $t_{1/2} = \ln 2 / k$; para orden 2, $t_{1/2} = 1/(k[A]_0)$.

<details><summary><b>Ver solución</b></summary>

```python
if ORDEN == 1:
    tabla_k["t_medio"] = np.log(2) / tabla_k["k"]
else:
    tabla_k["t_medio"] = 1 / (tabla_k["k"] * c0_aj)
print(tabla_k)
```
En orden 1 la vida media no depende de la concentración inicial; en orden 2 sí. Esa es la diferencia conceptual entre ambos.
</details>

**3.** Use la ecuación de Arrhenius para predecir $k$ a 70 °C y calcule cuánto tardaría la reacción en consumir el 90 % del reactivo a esa temperatura.

<details><summary><b>Ver solución</b></summary>

```python
T_nueva = 70 + 273.15
k_nueva = A * np.exp(-Ea / (R * T_nueva))
print("k a 70 °C:", k_nueva)
if ORDEN == 1:
    print("tiempo al 90 %:", np.log(10) / k_nueva, "min")
```
Es una extrapolación: solo es válida si el mecanismo no cambia en ese intervalo. Dígalo en el informe.
</details>

**4.** Repita el ajuste de Arrhenius usando solo tres de las cuatro temperaturas. ¿Cuánto cambia la $E_a$? Compare ese cambio con la incertidumbre que calculó.

<details><summary><b>Ver solución</b></summary>

```python
for fuera in range(len(tabla_k)):
    sub = tabla_k.drop(index=fuera)
    a = linregress(1/sub["T_K"], np.log(sub["k"]))
    print(f"sin {tabla_k.loc[fuera,'T_C']:.0f} °C -> Ea = {-a.slope*R:.1f} kJ/mol")
```
Si quitar un punto cambia la Ea más que su incertidumbre, el valor no es tan sólido como parecía.
</details>